In [1]:
import pandas as pd
import pymysql
from tqdm import tqdm
import os
import pickle
import numpy as np
def connect_db():
    return pymysql.connect(
        host="127.0.0.1",
        user='root',
        password='240812',
        database='lol_data',
        port=3307
    )

# DB 연결 함수

# 챔피언 ID, 태그 사전 파일 저장 함수

In [68]:
def get_champion_dict(conn):
    champion_dict_query = "SELECT champion_id, champion_name, tags FROM champion_dict"
    
    champion_dict = pd.read_sql_query(champion_dict_query, conn)

    with open("table_data/champion_dict.pkl", "wb") as f:
        pickle.dump(champion_dict, f)
    return champion_dict
conn = connect_db()
get_champion_dict(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\89247483.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)


# 챔피언 참여자ID 라인타입 사전 파일 저장 함수

In [18]:
def get_champion_participant_id(conn):
    champion_participant_id_query = "SELECT match_id, participant_id, champion_id, position FROM champion_participant_id"
    champion_participant_id = pd.read_sql_query(champion_participant_id_query, conn)
    with open("table_data/champion_participant_id.pkl", "wb") as f:
        pickle.dump(champion_participant_id, f)
    return champion_participant_id
conn = connect_db()
get_champion_participant_id(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_6472\3442806052.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_participant_id = pd.read_sql_query(champion_participant_id_query, conn)


# 게임 종료 파일 저장 함수

In [6]:
def get_game_end(conn):
    game_end_query = "SELECT match_id, real_timestamp, timestamp, winning_team FROM game_end"
    game_end = pd.read_sql_query(game_end_query, conn)
    with open("table_data/game_end.pkl", "wb") as f:
        pickle.dump(game_end, f)
    return game_end
conn = connect_db()
get_game_end(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_6472\226839149.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  game_end = pd.read_sql_query(game_end_query, conn)


# 타워 파괴 파일 저장 함수

In [ ]:
def get_building_kill(conn):
    building_kill_query = "SELECT match_id, assist_id, building_type, participant_id, line_type, team_id, timestamp, tower_type FROM building_kill"
    building_kill = pd.read_sql_query(building_kill_query, conn)
    with open("table_data/building_kill.pkl", "wb") as f:
        pickle.dump(building_kill, f)
    return building_kill
conn = connect_db()
get_building_kill(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_10216\2079252993.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  building_kill = pd.read_sql_query(building_kill_query, conn)


# 챔피언 20분 이후 솔킬 파일 저장함수

In [41]:
def get_champion_kill(conn):
    champion_kill_query = "SELECT match_id, participant_id, timestamp FROM champion_kill where assist_id IS NULL AND timestamp > 1200000"
    champion_kill = pd.read_sql_query(champion_kill_query, conn)
    with open("table_data/champion_solo_kill_after20.pkl", "wb") as f:
        pickle.dump(champion_kill, f)
    return champion_kill
conn = connect_db()
get_champion_kill(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_10216\1483398865.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_kill = pd.read_sql_query(champion_kill_query, conn)


# 6.태그별 평균 수치 데이터 파일 저장함수

In [71]:
def tag_avg_stat():
    with open("dashboard_data/champion_stats.pkl", "rb") as f:
        champion_dataframes = pickle.load(f)

    with open("table_data/champion_dict.pkl", "rb") as f:
        champion_df = pickle.load(f)
    tag_mapping = {
        "Assassin": "암살자",
        "Fighter": "전사",
        "Mage": "마법사",
        "Marksman": "원거리 공격",
        "Support": "지원가",
        "Tank": "탱커"
    }

    tag_avg_dataframes = {}
    for tag in set(tag for tags in champion_df['tags'] for tag in eval(tags)):

        champions_with_tag = champion_df[champion_df['tags'].apply(lambda x: tag in eval(x))]['champion_name'].tolist()
        
        tag_data = pd.concat([champion_dataframes[champion] for champion in champions_with_tag if champion in champion_dataframes])
        
        tag_avg_df = tag_data.groupby("분").mean().reset_index()


        mapped_tag = tag_mapping.get(tag, tag)  
        tag_avg_dataframes[mapped_tag] = tag_avg_df
    with open("dashboard_data/tag_avg_stats.pkl", "wb") as f:
        pickle.dump(tag_avg_dataframes, f)
tag_avg_stat()

In [72]:
with open("dashboard_data/tag_avg_stats.pkl", "rb") as f:
    data = pickle.load(f)
data['암살자']

,분,챔피언에게 가한 피해량,챔피언에게 가한 마법 피해량,챔피언에게 가한 물리 피해량,받은 피해량,총 획득한 골드량,총 경험치량,평균 누적 와드 설치 횟수,평균 누적 와드 제거 횟수
0,0,0.000000,0.000000,0.000000,0.000000,500.020455,0.000000,1.001384,1.001105
1,1,37.186364,8.143182,27.813636,40.288636,509.113636,0.381818,1.012561,1.022886
2,2,206.393182,41.063636,157.834091,452.256818,613.452273,216.084091,1.057348,1.091973
3,3,513.884091,119.375000,367.595455,1109.411364,979.209091,741.240909,1.328070,1.064068
4,4,884.111364,217.863636,610.302273,1699.286364,1352.329545,1171.572727,2.003420,1.122814
5,5,1268.072727,324.240909,861.072727,2301.404545,1725.447727,1617.768182,2.285352,1.150950
6,6,1652.159091,433.979545,1110.052273,2941.115909,2094.306818,2068.059091,2.660780,1.200677
7,7,2096.213636,565.215909,1393.902273,3673.561364,2463.679545,2492.713636,3.346120,1.306948
8,8,2563.979545,703.752273,1691.302273,4428.811364,2865.702273,2974.040909,3.716641,1.438280
9,9,3069.384091,855.781818,2006.813636,5219.784091,3272.034091,3453.263636,4.208959,1.579148


# 5.이벤트 파일 합치는 함수

In [59]:
def load_and_merge_champion_data():
    # 챔피언 데이터 로드
    with open("table_data/champion_main_stats.pkl", "rb") as f:
        champion_main_stats = pickle.load(f)
    
    with open("table_data/champion_ward_placement_stats.pkl", "rb") as f:
        champion_ward_placement_stats = pickle.load(f)
    
    with open("table_data/champion_ward_removal_stats.pkl", "rb") as f:
        champion_ward_removal_stats = pickle.load(f)
    
    merged_data = {}

    for champion_name in champion_main_stats.keys():
        main_stats = champion_main_stats.get(champion_name)
        placement_stats = champion_ward_placement_stats.get(champion_name)
        removal_stats = champion_ward_removal_stats.get(champion_name)
        
        # 분 기준으로 병합
        merged_df = pd.merge(main_stats, placement_stats, on='분', how='outer')
        merged_df = pd.merge(merged_df, removal_stats, on='분', how='outer')
        
        # 분 기준으로 정렬
        merged_df = merged_df.sort_values('분').reset_index(drop=True)
        
        # 병합된 데이터프레임을 딕셔너리에 저장
        merged_data[champion_name] = merged_df
    
    with open("dashboard_data/champion_stats.pkl", "wb") as f:
        pickle.dump(merged_data, f)

In [60]:
load_and_merge_champion_data()

In [61]:
with open("dashboard_data/champion_stats.pkl", "rb") as f:
    data = pickle.load(f)
data['카타리나']

,분,챔피언에게 가한 피해량,챔피언에게 가한 마법 피해량,챔피언에게 가한 물리 피해량,받은 피해량,총 획득한 골드량,총 경험치량,평균 누적 와드 설치 횟수,평균 누적 와드 제거 횟수
0,0,0.0,0.0,0.0,0.0,500.0,0.0,1.0000,1.0000
1,1,15.1,10.5,4.2,23.1,504.5,0.2,1.0000,1.0000
2,2,118.4,79.3,36.1,202.3,585.6,191.8,1.0095,NaN
3,3,433.0,273.5,128.2,689.9,896.2,821.9,1.0473,1.1667
4,4,857.1,550.6,233.0,1227.7,1250.2,1337.7,1.8186,1.0244
5,5,1282.8,843.4,326.4,1793.8,1636.6,1869.8,2.0039,1.0135
6,6,1723.0,1165.4,405.6,2353.3,2027.2,2409.2,2.1989,1.0769
7,7,2218.1,1552.3,473.8,2914.5,2410.1,2866.6,2.9676,1.1644
8,8,2744.1,1964.8,545.8,3510.1,2841.2,3412.1,3.1664,1.1570
9,9,3264.1,2378.7,611.0,4103.2,3265.4,3939.6,3.4518,1.3017


# 대쉬보드

## 1. 챔피언 기본 정보 저장 함수

In [17]:
# 챔피언 기본 스탯 추출 쿼리
def get_champion_main_stats(conn):
    main_stats_query = """
        SELECT 
            cp.champion_id,
            FLOOR(cs.timestamp / 60000) AS minute,
            ROUND(SUM(cs.tdd_to_champion) / COUNT(DISTINCT cs.match_id), 1) AS avg_tdd_to_champion,
            ROUND(SUM(cs.mdd_to_champion) / COUNT(DISTINCT cs.match_id), 1) AS avg_mdd_to_champion,
            ROUND(SUM(cs.pdd_to_champion) / COUNT(DISTINCT cs.match_id), 1) AS avg_pdd_to_champion,
            ROUND(SUM(cs.total_damage_taken) / COUNT(DISTINCT cs.match_id), 1) AS avg_damage_taken,
            ROUND(SUM(cs.total_gold) / COUNT(DISTINCT cs.match_id), 1) AS avg_gold,
            ROUND(SUM(cs.xp) / COUNT(DISTINCT cs.match_id), 1) AS avg_xp
        FROM 
            champion_stat_per_timestamp cs
        JOIN 
            champion_participant_id cp ON cs.match_id = cp.match_id AND cs.participant_id = cp.participant_id
        GROUP BY 
            cp.champion_id, FLOOR(cs.timestamp / 60000)
        ORDER BY 
            cp.champion_id, minute;
    """
    return pd.read_sql_query(main_stats_query, conn)

In [32]:
def save_main_stats_to_pkl():

    conn = connect_db()
    with open("table_data/champion_dict.pkl", "rb") as f:
        champion_dict = pickle.load(f) 

    main_stats = get_champion_main_stats(conn)
    conn.close()

    column_mapping = {
        "minute": "분",
        "avg_tdd_to_champion": "챔피언에게 가한 피해량",
        "avg_mdd_to_champion": "챔피언에게 가한 마법 피해량",
        "avg_pdd_to_champion": "챔피언에게 가한 물리 피해량",
        "avg_damage_taken": "받은 피해량",
        "avg_gold": "총 획득한 골드량",
        "avg_xp": "총 경험치량",
    }

    champion_dataframes = {}

    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        champion_df = main_stats[main_stats['champion_id'] == champion_id].drop(columns=['champion_id'])
        champion_df = champion_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = champion_df
    with open("table_data/champion_main_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [33]:
save_main_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\3419427336.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(main_stats_query, conn)
Processing Champions: 100%|██████████| 168/168 [00:00<00:00, 915.49it/s]


In [34]:
with open("table_data/champion_main_stats.pkl", "rb") as f:
    champion_main_stats_data = pickle.load(f)
champion_main_stats_data['카타리나']

,분,챔피언에게 가한 피해량,챔피언에게 가한 마법 피해량,챔피언에게 가한 물리 피해량,받은 피해량,총 획득한 골드량,총 경험치량
2479,0,0.0,0.0,0.0,0.0,500.0,0.0
2480,1,15.1,10.5,4.2,23.1,504.5,0.2
2481,2,118.4,79.3,36.1,202.3,585.6,191.8
2482,3,433.0,273.5,128.2,689.9,896.2,821.9
2483,4,857.1,550.6,233.0,1227.7,1250.2,1337.7
2484,5,1282.8,843.4,326.4,1793.8,1636.6,1869.8
2485,6,1723.0,1165.4,405.6,2353.3,2027.2,2409.2
2486,7,2218.1,1552.3,473.8,2914.5,2410.1,2866.6
2487,8,2744.1,1964.8,545.8,3510.1,2841.2,3412.1
2488,9,3264.1,2378.7,611.0,4103.2,3265.4,3939.6


## 2. 와드

In [39]:
# 와드 제거 횟수 데이터 추출
def get_ward_removal_stats(conn):
    ward_removal_query = """
    WITH filtered_ward_kills AS (
        SELECT 
            wk.match_id,
            cp.champion_id,
            FLOOR(wk.timestamp / 60000) AS minute,
            COUNT(*) AS ward_removals
        FROM 
            ward_kill wk
        JOIN 
            champion_participant_id cp ON wk.match_id = cp.match_id AND wk.participant_id = cp.participant_id
        WHERE 
            wk.ward_type != 'UNDEFINED'
        GROUP BY 
            wk.match_id, cp.champion_id, minute
    ),
    cumulative_ward_removals AS (
        SELECT 
            match_id,
            champion_id,
            minute,
            SUM(ward_removals) OVER (PARTITION BY match_id, champion_id ORDER BY minute ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_removals
        FROM 
            filtered_ward_kills
    )
    SELECT 
        champion_id,
        minute,
        AVG(cumulative_removals) AS avg_cumulative_ward_removals
    FROM 
        cumulative_ward_removals
    GROUP BY 
        champion_id, minute
    ORDER BY 
        champion_id, minute;
    """
    return pd.read_sql_query(ward_removal_query, conn)

# 와드 설치 횟수 데이터 추출
def get_ward_placement_stats(conn):
    ward_placement_query = """
    WITH filtered_ward_placements AS (
        SELECT 
            wp.match_id,
            cp.champion_id,
            FLOOR(wp.timestamp / 60000) AS minute,
            COUNT(*) AS ward_placements
        FROM 
            ward_placed wp
        JOIN 
            champion_participant_id cp ON wp.match_id = cp.match_id AND wp.participant_id = cp.participant_id
        GROUP BY 
            wp.match_id, cp.champion_id, minute
    ),
    cumulative_ward_placements AS (
        SELECT 
            match_id,
            champion_id,
            minute,
            SUM(ward_placements) OVER (PARTITION BY match_id, champion_id ORDER BY minute ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_placements
        FROM 
            filtered_ward_placements
    )
    SELECT 
        champion_id,
        minute,
        AVG(cumulative_placements) AS avg_cumulative_ward_placements
    FROM 
        cumulative_ward_placements
    GROUP BY 
        champion_id, minute
    ORDER BY 
        champion_id, minute;
    """
    return pd.read_sql_query(ward_placement_query, conn)

def save_ward_removal_stats_to_pkl():
    conn = connect_db()
    ward_removal_stats = get_ward_removal_stats(conn)
    ward_placement_stats = get_ward_placement_stats(conn)
    conn.close()  
    with open("table_data/ward_removal_stats.pkl", "wb") as f:
        pickle.dump(ward_removal_stats, f)
    with open("table_data/ward_placement_stats.pkl", "wb") as f:
        pickle.dump(ward_placement_stats, f)

In [40]:
save_ward_removal_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\2253587392.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(ward_removal_query, conn)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\2253587392.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(ward_placement_query, conn)


### 3. 챔피언별 와드제거 데이터 저장

In [51]:
def save_champion_ward_removal_stats_to_pkl():
    conn = connect_db()
    champion_dict = get_champion_dict(conn)
    conn.close()  
    with open("table_data/ward_removal_stats.pkl", "rb") as f:
        ward_removal_stats = pickle.load(f)
    column_mapping = {
        "minute": "분",
        "avg_cumulative_ward_removals": "평균 누적 와드 제거 횟수",
    }
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        removal_df = ward_removal_stats[ward_removal_stats['champion_id'] == champion_id][['minute', 'avg_cumulative_ward_removals']]
        removal_df = removal_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = removal_df
    with open("table_data/champion_ward_removal_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [52]:
save_champion_ward_removal_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\2181235943.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)
Processing Champions: 100%|██████████| 168/168 [00:00<00:00, 971.05it/s]


In [48]:
with open("dashboard_data/champion_ward_removal_stats.pkl", "rb") as f:
    ward_removal_stats = pickle.load(f)
ward_removal_stats['카타리나'].isnull().sum()

분                 0
평균 누적 와드 제거 횟수    0
dtype: int64

### 4. 챔피언별 와드 설치 데이터 저장

In [53]:
def save_champion_ward_placement_stats_to_pkl():
    conn = connect_db()
    champion_dict = get_champion_dict(conn)
    conn.close()  
    with open("table_data/ward_placement_stats.pkl", "rb") as f:
        ward_placement_stats = pickle.load(f)
    column_mapping = {
        "minute": "분",
        "avg_cumulative_ward_placements": "평균 누적 와드 설치 횟수",
    }
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        placement_df = ward_placement_stats[ward_placement_stats['champion_id'] == champion_id][['minute', 'avg_cumulative_ward_placements']]
        placement_df = placement_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = placement_df
    with open("table_data/champion_ward_placement_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [54]:
save_champion_ward_placement_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\2181235943.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)
Processing Champions: 100%|██████████| 168/168 [00:00<00:00, 988.38it/s]


In [39]:
with open('dashboard_data/champion_main_stats.pkl','rb') as f:
    data = pickle.load(f)
data['닐라'].head(5)

,분,챔피언에게 가한 피해량,받은 피해량,총 획득한 골드량,총 경험치량
8116,0,0.0,0.0,500.0,0.0
8117,1,34.2,35.2,509.9,0.4
8118,2,200.7,291.4,601.6,116.0
8119,3,444.2,726.2,962.4,577.5
8120,4,783.0,1158.6,1360.6,988.3


## 챔피언 승률

In [40]:
def get_champion_win_pick_rate():
    with open('table_data/champion_participant_id.pkl', 'rb') as f:
        champion_participant_id = pickle.load(f)
    with open('table_data/game_end.pkl', 'rb') as f:
        game_end = pickle.load(f)
    with open('table_data/champion_dict.pkl', 'rb') as f:
        champion_dict = pickle.load(f)

    # 데이터 복사
    champion_participant_df = champion_participant_id.copy()
    game_end_df = game_end.copy()
    champion_dict_df = champion_dict[['champion_id', 'champion_name']]

    # 팀 정보 추가
    champion_participant_df['team'] = champion_participant_df['participant_id'].apply(lambda x: 100 if x <= 5 else 200)

    # game_end_df와 병합하여 각 챔피언이 승리했는지 여부 계산
    merged_df = champion_participant_df.merge(game_end_df[['match_id', 'winning_team']], on='match_id')
    merged_df['win'] = merged_df['team'] == merged_df['winning_team']

    # 각 position, champion_id별로 승리 횟수, 총 픽 횟수 집계
    grouped_df = merged_df.groupby(['position', 'champion_id']).agg(
        total_picks=('match_id', 'count'),
        wins=('win', 'sum')
    ).reset_index()

    # 포지션별 경기 수 집계
    position_match_counts = merged_df.groupby('position')['match_id'].nunique().reset_index(name='position_match_counts')
    grouped_df = grouped_df.merge(position_match_counts, on='position')

    # 승률 및 픽률 계산
    grouped_df['win_rate'] = (grouped_df['wins'] / grouped_df['total_picks']).round(4)
    grouped_df['pick_rate'] = (grouped_df['total_picks'] / grouped_df['position_match_counts']).round(4)

    # 챔피언 이름을 추가하기 위해 champion_dict와 병합
    grouped_df = grouped_df.merge(champion_dict_df, on='champion_id')

    # 최종 결과물 구성, 컬럼 이름 수정 및 필요 없는 컬럼 제거
    result_df = grouped_df[['champion_name', 'position', 'win_rate', 'pick_rate', 'total_picks']]
    result_df.columns = ['챔피언', '포지션', '승률', '픽률', '표본 수']

    # 포지션별로 데이터프레임을 분리하여 딕셔너리에 저장
    position_dict = {position: df for position, df in result_df.groupby(result_df['포지션'])}

    # **전체 포지션('ALL') 승률 및 픽률 계산**
    # 전체 픽 수와 승리 수를 챔피언별로 집계
    total_grouped_df = merged_df.groupby('champion_id').agg(
        total_picks=('match_id', 'count'),
        wins=('win', 'sum')
    ).reset_index()

    # 전체 경기 수 계산 (모든 포지션을 포함하여 고유한 경기 수)
    total_matches = merged_df['match_id'].nunique()

    # 전체 승률 및 픽률 계산
    total_grouped_df['win_rate'] = (total_grouped_df['wins'] / total_grouped_df['total_picks']).round(4)
    total_grouped_df['pick_rate'] = (total_grouped_df['total_picks'] / total_matches).round(4)

    # 챔피언 이름을 추가하기 위해 champion_dict와 병합
    total_grouped_df = total_grouped_df.merge(champion_dict_df, on='champion_id')

    # **'ALL' 포지션 데이터프레임 구성**
    all_position_df = total_grouped_df[['champion_name', 'win_rate', 'pick_rate', 'total_picks']]
    all_position_df.columns = ['챔피언', '승률', '픽률', '표본 수']

    # 'ALL' 키로 추가
    position_dict['ALL'] = all_position_df

    # 딕셔너리를 파일로 저장
    with open("dashboard_data/champion_win_pick_rate.pkl", "wb") as f:
        pickle.dump(position_dict, f)

In [41]:
get_champion_win_pick_rate()

In [9]:
with open('dashboard_data/champion_win_pick_rate.pkl', 'rb') as f:
    champion_win_pick_rate = pickle.load(f)

In [10]:
champion_win_pick_rate['ALL'].sort_values('픽률')

,챔피언,승률,픽률,표본 수
85,쉬바나,0.4851,0.0034,202
9,케일,0.4351,0.0044,262
45,트런들,0.4590,0.0051,305
36,소나,0.4756,0.0055,328
167,나피리,0.4668,0.0063,377
...,...,...,...,...
59,리 신,0.4885,0.2376,14213
41,코르키,0.5197,0.2432,14550
87,그레이브즈,0.5056,0.2716,16247
128,비에고,0.4980,0.2794,16718


## 챔피언 상대 챔피언별 승률

In [77]:
def get_winrate_per_champion():
    conn = connect_db()
    query = "SELECT champion, opponent_champion, winrate, game_count FROM winrate_per_champion"
    df = pd.read_sql(query, conn)
    conn.close()

    # 2. 챔피언별로 상대 챔피언 승률 및 표본 수 딕셔너리 생성
    champion_vs_champion_winrate_dict = {}

    for champion, group in df.groupby('champion'):
        # 컬럼 이름 변경 및 인덱스 새로 설정
        opponent_winrate_df = group[['opponent_champion', 'winrate', 'game_count']].rename(
            columns={'opponent_champion': '상대 챔피언', 'winrate': '승률', 'game_count': '표본 수'}
        )
        opponent_winrate_df = opponent_winrate_df.sort_values(by='승률', ascending=False).reset_index(drop=True)

        # 딕셔너리에 저장
        champion_vs_champion_winrate_dict[champion] = opponent_winrate_df

    # 결과를 pickle 파일로 저장
    output_file_path = "dashboard_data/winrate_per_champion.pkl"
    with open(output_file_path, "wb") as f:
        pickle.dump(champion_vs_champion_winrate_dict, f)

In [78]:
get_winrate_per_champion()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_2912\1845079828.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [79]:
with open('dashboard_data/winrate_per_champion.pkl', 'rb') as f:
    winrate_per_champion = pickle.load(f)

In [80]:
winrate_per_champion['가렌']

,상대 챔피언,승률,표본 수
0,말자하,1.0,2
1,아무무,1.0,8
2,루시안,1.0,16
3,밀리오,1.0,4
4,초가스,1.0,12
...,...,...,...
161,녹턴,0.0,8
162,아우렐리온 솔,0.0,4
163,요릭,0.0,4
164,브라이어,0.0,4


In [88]:
with open('dashboard_data/champion_win_pick_rate.pkl', 'rb') as f:
    champion_win_pick_rate = pickle.load(f)
champion_win_pick_rate['BOTTOM']

,챔피언,포지션,승률,픽률,표본 수
0,애니,BOTTOM,0.7500,0.0001,4
1,갈리오,BOTTOM,0.2500,0.0001,4
2,트위스티드 페이트,BOTTOM,0.4615,0.0002,13
3,우르곳,BOTTOM,1.0000,0.0000,1
4,블라디미르,BOTTOM,0.6250,0.0001,8
...,...,...,...,...,...
117,오로라,BOTTOM,0.4000,0.0002,10
118,닐라,BOTTOM,0.4990,0.0081,487
119,크산테,BOTTOM,0.5532,0.0008,47
120,스몰더,BOTTOM,0.4802,0.0186,1112


## 챔피언별 역할군 상대 승률

In [ ]:
def get_champion_winrate_by_role():
    # 데이터 불러오기
    with open('table_data/champion_participant_id.pkl', 'rb') as f:
        champion_participant_id = pickle.load(f)
    with open('table_data/game_end.pkl', 'rb') as f:
        game_end = pickle.load(f)
    with open('table_data/champion_dict.pkl', 'rb') as f:
        champion_dict = pickle.load(f)

    # 중복 제거된 game_end_unique 생성
    game_end_unique = game_end.drop_duplicates(subset=['match_id', 'winning_team'])

    # 역할군 태그 매핑
    tag_mapping = {
        "Assassin": "암살자",
        "Fighter": "전사",
        "Mage": "마법사",
        "Marksman": "원거리 공격",
        "Support": "지원가",
        "Tank": "탱커"
    }

    # 챔피언별 역할군 승률 데이터 딕셔너리
    champion_winrate_dict = {}

    # 각 챔피언에 대해 역할군 승률 계산
    for idx, row in champion_dict.iterrows():
        champion_id = row['champion_id']
        champion_name = row['champion_name']

        # 챔피언의 경기를 필터링
        champion_matches = champion_participant_id[champion_participant_id['champion_id'] == champion_id]['match_id'].unique()
        champion_match_data = champion_participant_id[champion_participant_id['match_id'].isin(champion_matches)][['match_id', 'participant_id', 'champion_id']]
        
        # 챔피언의 역할군 데이터 추가
        champion_match_data = champion_match_data.merge(champion_dict[['champion_id', 'tags']], on='champion_id', how='left')
        
        # 경기 종료 데이터와 결합하여 승리 팀 정보 추가
        champion_match_data = champion_match_data.merge(game_end_unique[['match_id', 'winning_team']], on='match_id', how='left')
        champion_match_data['participant_team'] = champion_match_data['participant_id'].map(lambda x: 100 if x <= 5 else 200)
        
        # 챔피언의 팀 식별
        champion_team_data = champion_match_data[champion_match_data['champion_id'] == champion_id]
        champion_team_data['team'] = champion_team_data['participant_id'].map(lambda x: 100 if x <= 5 else 200)
        
        # 상대 팀 데이터 필터링
        opponent_team_data = champion_match_data[
            champion_match_data\
            .merge(champion_team_data, on='match_id', how='left')[['match_id', 'participant_team_x', 'champion_id_x', 'tags_x', 'winning_team_x', 'team']]
            .apply(lambda row: (row['participant_team_x'] != row['team']), axis=1)
        ]

        # 역할군별 데이터프레임 딕셔너리 생성
        role_winrate_dict = {}

        for target_role_eng, target_role_kor in tag_mapping.items():
            # 상대팀에서 각 챔피언이 특정 역할군(target_role_eng)인지 판단
            opponent_team_data['is_target_role'] = opponent_team_data['tags'].apply(lambda tags: target_role_eng in tags)

            # 각 경기별로 특정 역할군의 수 계산
            role_count_per_match = opponent_team_data.groupby('match_id')['is_target_role'].sum().reset_index()
            role_count_per_match.columns = ['match_id', 'target_role_count']

            # 역할군 명수별 승률 및 표본 수 계산
            win_rates = []
            for count in range(6):  # 0명부터 5명까지의 역할군 명수
                # 특정 역할군 수를 포함한 경기들의 match_id 추출
                role_match_ids = role_count_per_match[role_count_per_match['target_role_count'] == count]['match_id']
                role_match_results = opponent_team_data[opponent_team_data['match_id'].isin(role_match_ids)]

                # 승리 여부 계산
                role_match_results['champion_win'] = role_match_results.apply(
                    lambda row: 1 if row['participant_team'] != row['winning_team'] else 0, axis=1
                )

                # 각 경기별 승/패 결과 중 한 경기당 하나의 값만 남기기 위해 drop_duplicates 사용
                role_match_results = role_match_results.drop_duplicates(subset=['match_id'])

                # 승률 및 표본 수 계산
                sample_size = role_match_results.shape[0]
                if sample_size > 0:
                    win_rate = role_match_results['champion_win'].mean()
                else:
                    win_rate = np.nan

                win_rates.append({'Count': count, '승률': win_rate, '표본 수': sample_size})

            # 역할군별 데이터프레임 생성 후 딕셔너리에 추가
            role_winrate_dict[target_role_kor] = pd.DataFrame(win_rates).set_index('Count')

        # 챔피언 이름을 키로 하고, 역할군 승률 딕셔너리를 값으로 추가
        champion_winrate_dict[champion_name] = role_winrate_dict

    # 딕셔너리를 파일로 저장
    with open("dashboard_data/champion_winrate_by_role.pkl", "wb") as f:
        pickle.dump(champion_winrate_dict, f)


C:\Users\dgjja\AppData\Local\Temp\ipykernel_15916\2665691659.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  champion_team_data['team'] = champion_team_data['participant_id'].map(lambda x: 100 if x <= 5 else 200)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_15916\2665691659.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  opponent_team_data['is_target_role'] = opponent_team_data['tags'].apply(lambda tags: target_role_eng in tags)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_15916\2665691659.py:71: Se

In [79]:
with open('dashboard_data/champion_winrate_by_role.pkl', 'rb') as f:
    champion_winrate_by_role = pickle.load(f)

In [80]:
champion_winrate_by_role['워윅']['암살자']

,승률,표본 수
Count,,
0,0.520958,334
1,0.504255,940
2,0.521333,750
3,0.530000,200
4,0.518519,27
5,1.000000,1


## 사이드 타워 파괴 횟수

In [147]:
def get_side_building_kill_per_champion():
    with open('table_data/building_kill.pkl', 'rb') as f:
        building_kill = pickle.load(f)
    with open('table_data/champion_participant_id.pkl', 'rb') as f:
        champion_participant_id = pickle.load(f)
    with open('table_data/champion_dict.pkl', 'rb') as f:
        champion_dict = pickle.load(f)

    merged_df = building_kill.merge(champion_participant_id, on=["match_id", "participant_id"], how="left")
    filtered_df = merged_df[merged_df["line_type"].isin(["BOT_LANE", "TOP_LANE"])]
    filtered_df = filtered_df.dropna(subset=["champion_id"])
    per_game_counts = (
        filtered_df.groupby(["match_id", "champion_id"])
        .size()
        .reset_index(name="타워 파괴 횟수")
    )
    building_kill_mean = (
        per_game_counts.groupby("champion_id")["타워 파괴 횟수"]
        .mean()
        .reset_index(name="평균 타워 파괴 횟수")
    )
    building_kill_mean["champion_id"] = building_kill_mean["champion_id"].astype(int)
    champion_dict = champion_dict[["champion_id", "champion_name"]] 
    building_kill_mean = building_kill_mean.merge(champion_dict, on="champion_id", how="left")
    building_kill_mean = building_kill_mean.drop(columns=["champion_id"])
    building_kill_mean = building_kill_mean.rename(columns={"champion_name": "챔피언"})
    building_kill_mean = building_kill_mean[["챔피언", "평균 타워 파괴 횟수"]]
    building_kill_mean = building_kill_mean.sort_values(by="평균 타워 파괴 횟수", ascending=False).reset_index(drop=True)
    
    with open("dashboard_data/champion_sidebuilding.pkl", "wb") as f:
        pickle.dump(building_kill_mean, f)

In [148]:
get_side_building_kill_per_champion()

In [149]:
with open('dashboard_data/champion_sidebuilding.pkl', 'rb') as f:
    building_kill_mean = pickle.load(f)

In [150]:
building_kill_mean

,챔피언,평균 타워 파괴 횟수
0,트런들,2.553922
1,피오라,2.241059
2,트린다미어,2.161754
3,요릭,2.117096
4,나서스,1.939971
...,...,...
163,레오나,1.045181
164,블리츠크랭크,1.042857
165,모르가나,1.037037
166,유미,1.018868


## 챔피언 20분 이후 솔로킬 횟수

In [102]:
def get_solokill_per_champion():
    with open('table_data/champion_solo_kill_after20.pkl', 'rb') as f:
        champion_kill = pickle.load(f)
    with open('table_data/champion_participant_id.pkl', 'rb') as f:
        champion_participant_id = pickle.load(f)
    with open('table_data/champion_dict.pkl', 'rb') as f:
        champion_dict = pickle.load(f)

    merged_df = champion_kill.merge(champion_participant_id, on=["match_id", "participant_id"], how="left")
    champion_match_kill_count = merged_df.groupby(['champion_id', 'match_id']).size().reset_index(name='solo_kill_count')
    champion_avg_solo_kill = champion_match_kill_count.groupby('champion_id')['solo_kill_count'].mean().reset_index()
    champion_avg_solo_kill.rename(columns={'solo_kill_count': '20분 이후 평균 솔로킬'}, inplace=True)
    champion_avg_solo_kill["champion_id"] = champion_avg_solo_kill["champion_id"].astype(int)    
    champion_dict = champion_dict[["champion_id", "champion_name"]] 
    champion_avg_solo_kill = champion_avg_solo_kill.merge(champion_dict, on="champion_id", how="left")
    champion_avg_solo_kill = champion_avg_solo_kill.drop(columns=["champion_id"])
    champion_avg_solo_kill = champion_avg_solo_kill.rename(columns={"champion_name": "챔피언"})
    champion_avg_solo_kill = champion_avg_solo_kill[["챔피언", "20분 이후 평균 솔로킬"]]
    champion_avg_solo_kill = champion_avg_solo_kill.sort_values(by="20분 이후 평균 솔로킬", ascending=False).reset_index(drop=True)
    
    with open("dashboard_data/champion_solokill_after20.pkl", "wb") as f:
        pickle.dump(champion_avg_solo_kill, f)

In [103]:
get_solokill_per_champion()

In [104]:
with open('dashboard_data/champion_solokill_after20.pkl', 'rb') as f:
    champion_solokill_after20 = pickle.load(f)

In [105]:
champion_solokill_after20

,챔피언,20분 이후 평균 솔로킬
0,나피리,1.854369
1,탈론,1.690967
2,퀸,1.614583
3,카사딘,1.596026
4,제드,1.565804
...,...,...
163,소라카,1.000000
164,아무무,1.000000
165,소나,1.000000
166,아이번,1.000000


In [108]:
def splitpush_champion():
    with open('dashboard_data/champion_sidebuilding.pkl', 'rb') as f:
        building_kill_mean = pickle.load(f)
    with open('dashboard_data/champion_solokill_after20.pkl', 'rb') as f:
        champion_solokill_after20 = pickle.load(f)

    merged_df = pd.merge(building_kill_mean, champion_solokill_after20, on="챔피언", how="outer")
    threshold_tower = merged_df["평균 타워 파괴 횟수"].quantile(0.80)
    threshold_solo_kill = merged_df["20분 이후 평균 솔로킬"].quantile(0.80)

    filtered_data = merged_df[
        (merged_df["평균 타워 파괴 횟수"] >= threshold_tower) &
        (merged_df["20분 이후 평균 솔로킬"] >= threshold_solo_kill)
    ]
    return filtered_data

In [109]:
splitpush_champion()

,챔피언,평균 타워 파괴 횟수,20분 이후 평균 솔로킬
5,그웬,0.044556,1.503344
34,리븐,0.037265,1.428058
88,아칼리,0.046868,1.528432
104,요네,0.135846,1.383550
123,제이스,0.080437,1.372068
156,트린다미어,0.048269,1.490099
163,피오라,0.081433,1.517287


# 스플릿 푸쉬 챔피언 구하기

In [ ]:
# 상위 20퍼